# Lectures 20, 21, 23: Gaussian Elimination, Pivoting & Cholesky
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy, SciPy, and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

---
## Lecture 20: Gaussian Elimination / LU Factorization

### Step-by-Step LU Example

In [ ]:
A = np.array([[2, 1, 1],
              [4, 3, 3],
              [8, 7, 9]], dtype=float)

# Step 1: eliminate below a_{11} = 2
L1 = np.eye(3)
L1[1, 0] = -A[1, 0] / A[0, 0]  # -2
L1[2, 0] = -A[2, 0] / A[0, 0]  # -4
A1 = L1 @ A
print("After step 1 (L1 @ A):")
print(A1.round(10))
print(f"Multipliers: l_21 = {-L1[1,0]}, l_31 = {-L1[2,0]}\n")

# Step 2: eliminate below a_{22}
L2 = np.eye(3)
L2[2, 1] = -A1[2, 1] / A1[1, 1]  # -3
U = L2 @ A1
print("After step 2 (U = L2 @ L1 @ A):")
print(U.round(10))
print(f"Multiplier: l_32 = {-L2[2,1]}\n")

# Assemble L from multipliers
L = np.linalg.inv(L1) @ np.linalg.inv(L2)
print("L (multipliers interleaved):")
print(L.round(10))
print(f"\n||LU - A|| = {np.linalg.norm(L @ U - A):.2e}")

### Solving $Ax = b$ via Forward/Back Substitution

In [ ]:
b = np.array([1.0, 1.0, 1.0])

# Forward substitution: Ly = b
y = np.zeros(3)
y[0] = b[0]
y[1] = b[1] - L[1, 0] * y[0]
y[2] = b[2] - L[2, 0] * y[0] - L[2, 1] * y[1]
print(f"Forward solve: y = {y}")

# Back substitution: Ux = y
x = np.zeros(3)
x[2] = y[2] / U[2, 2]
x[1] = (y[1] - U[1, 2] * x[2]) / U[1, 1]
x[0] = (y[0] - U[0, 1] * x[1] - U[0, 2] * x[2]) / U[0, 0]
print(f"Back solve:    x = {x}")
print(f"Check: Ax = {A @ x}")

### LU with SciPy

In [ ]:
P, L_sp, U_sp = linalg.lu(A)
print(f"P =\n{P}\n")
print(f"L =\n{L_sp.round(6)}\n")
print(f"U =\n{U_sp.round(6)}\n")
print(f"||PA - LU|| = {np.linalg.norm(P @ A - L_sp @ U_sp):.2e}")

### Instability Without Pivoting: Small Pivot Disaster

In [ ]:
def lu_no_pivot(A):
    """LU without pivoting."""
    n = A.shape[0]
    L = np.eye(n)
    U = A.astype(float).copy()
    for k in range(n - 1):
        for j in range(k + 1, n):
            L[j, k] = U[j, k] / U[k, k]
            U[j, k:] -= L[j, k] * U[k, k:]
    return L, U

eps = 1e-20
A_bad = np.array([[eps, 1],
                  [1,   1]])

L_np, U_np = lu_no_pivot(A_bad)
print(f"A = [[{eps}, 1], [1, 1]]")
print(f"\nWithout pivoting:")
print(f"  L = {L_np.tolist()}")
print(f"  U = {U_np.tolist()}")
print(f"  Multiplier = {L_np[1,0]:.0e}  (huge!)")

b = np.array([1.0, 2.0])
x_true = np.linalg.solve(A_bad, b)
y = linalg.solve_triangular(L_np, b, lower=True)
x_bad = linalg.solve_triangular(U_np, y)
print(f"\n  True x:     {x_true}")
print(f"  No-pivot x: {x_bad}")
print(f"  Error:      {np.linalg.norm(x_bad - x_true) / np.linalg.norm(x_true):.2e}")

# With pivoting (SciPy)
x_good = np.linalg.solve(A_bad, b)
print(f"\n  Pivoted x:  {x_good}")
print(f"  Error:      {np.linalg.norm(x_good - x_true) / np.linalg.norm(x_true):.2e}")

---
## Lecture 21: Pivoting

### LU with Partial Pivoting: Step by Step

In [ ]:
A = np.array([[17,  2,  3, 13],
              [ 5, 12, 10,  8],
              [ 9,  7,  7, 12],
              [ 4, 14, 15,  2]], dtype=float)

P, L, U = linalg.lu(A)
print("P (permutation):")
print(P.round(0).astype(int))
print(f"\nL (all |l_ij| <= 1):")
print(L.round(6))
print(f"\nMax |l_ij| below diagonal: {np.max(np.abs(np.tril(L, -1))):.6f}")
print(f"\nU:")
print(U.round(4))
print(f"\n||PA - LU|| = {np.linalg.norm(P @ A - L @ U):.2e}")

### The Wilkinson Matrix: Exponential Growth Factor

In [ ]:
def wilkinson_matrix(m):
    """Build the m x m Wilkinson growth-factor matrix."""
    A = np.tril(-np.ones((m, m)), -1) + np.eye(m)
    A[:, -1] = 1.0
    return A

# Show the structure
m = 6
A = wilkinson_matrix(m)
print(f"Wilkinson matrix ({m}x{m}):")
print(A.astype(int))

P, L, U = linalg.lu(A)
print(f"\nU (last column grows as powers of 2):")
print(U.round(0))
print(f"\nGrowth factor rho = {np.max(np.abs(U)):.0f} = 2^{np.log2(np.max(np.abs(U))):.0f}")

In [ ]:
# The disaster: m = 53 -> rho = 2^52 ≈ 1/eps
m = 53
A = wilkinson_matrix(m)
x_true = np.arange(1, m + 1, dtype=float) / m
b = A @ x_true

x_lu = np.linalg.solve(A, b)

# QR for comparison
Q, R = np.linalg.qr(A)
x_qr = np.linalg.solve(R, Q.T @ b)

print(f"Wilkinson matrix, m = {m}")
print(f"cond(A) = {np.linalg.cond(A):.2f}  (well-conditioned!)")
print(f"Growth factor = 2^{m-1} = {2**(m-1):.2e}")
print(f"\nLU relative error:  {np.linalg.norm(x_lu - x_true)/np.linalg.norm(x_true):.6e}")
print(f"QR relative error:  {np.linalg.norm(x_qr - x_true)/np.linalg.norm(x_true):.6e}")
print(f"\nLU fails despite excellent conditioning!")

### Growth Factors: Random Matrices vs. Theory

In [ ]:
np.random.seed(0)
sizes = [10, 20, 50, 100, 200, 500]
growth_random = []
growth_worst = []

for n in sizes:
    # Random matrix
    rhos = []
    for _ in range(20):
        A = np.random.randn(n, n)
        _, _, U = linalg.lu(A)
        rhos.append(np.max(np.abs(U)) / np.max(np.abs(A)))
    growth_random.append(np.median(rhos))

    # Worst case
    growth_worst.append(2.0**(n-1))

plt.figure(figsize=(8, 5))
plt.semilogy(sizes, growth_random, 'bo-', markersize=5, label='Random matrices (median)')
plt.semilogy(sizes, growth_worst, 'r--', alpha=0.5, label=r'Worst case $2^{n-1}$')
plt.semilogy(sizes, np.sqrt(sizes), 'g--', alpha=0.5, label=r'$\sqrt{n}$')
plt.xlabel('$n$')
plt.ylabel('Growth factor $\\rho_n$')
plt.title('Growth factor: practice vs. worst-case bound')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

### LU vs. QR: Accuracy for Square Systems

In [ ]:
np.random.seed(1)
n = 50
kappas = np.logspace(1, 14, 20)

err_lu = []
err_qr = []

for kappa in kappas:
    U_rand, _ = np.linalg.qr(np.random.randn(n, n))
    V_rand, _ = np.linalg.qr(np.random.randn(n, n))
    s = np.logspace(0, -np.log10(kappa), n)
    A = U_rand @ np.diag(s) @ V_rand.T
    x_true = np.random.randn(n)
    b = A @ x_true

    x_l = np.linalg.solve(A, b)
    Q, R = np.linalg.qr(A)
    x_q = np.linalg.solve(R, Q.T @ b)

    err_lu.append(np.linalg.norm(x_l - x_true) / np.linalg.norm(x_true))
    err_qr.append(np.linalg.norm(x_q - x_true) / np.linalg.norm(x_true))

eps = np.finfo(float).eps

plt.figure(figsize=(8, 5))
plt.loglog(kappas, err_lu, 'bo-', markersize=4, label='LU (pivoted)')
plt.loglog(kappas, err_qr, 'rs-', markersize=4, label='Householder QR')
plt.loglog(kappas, kappas * eps, 'k--', alpha=0.4, label=r'$\kappa \cdot \epsilon$')
plt.xlabel(r'$\kappa(A)$')
plt.ylabel('Relative error')
plt.title('LU vs QR for square systems (both achieve $\\kappa \\cdot \\epsilon$)')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

---
## Lecture 23: Cholesky Factorization

### Cholesky: Step by Step

In [ ]:
A = np.array([[4, 2, 0],
              [2, 5, 3],
              [0, 3, 13]], dtype=float)

print(f"A =\n{A}")
print(f"Eigenvalues: {np.linalg.eigvalsh(A).round(4)}  (all positive -> SPD)\n")

# Manual Cholesky
n = 3
R = np.zeros((n, n))

# Step 1
R[0, 0] = np.sqrt(A[0, 0])
R[0, 1] = A[0, 1] / R[0, 0]
R[0, 2] = A[0, 2] / R[0, 0]
print(f"Step 1: R[0,:] = {R[0,:]}")

# Step 2: Schur complement
A1 = A[1:, 1:] - np.outer(R[0, 1:], R[0, 1:])
print(f"Schur complement: {A1.tolist()}")
R[1, 1] = np.sqrt(A1[0, 0])
R[1, 2] = A1[0, 1] / R[1, 1]

# Step 3
R[2, 2] = np.sqrt(A1[1, 1] - R[1, 2]**2)

print(f"\nR =\n{R.round(6)}")
print(f"\n||R^T R - A|| = {np.linalg.norm(R.T @ R - A):.2e}")

# Compare with NumPy
R_np = np.linalg.cholesky(A).T  # NumPy returns lower; transpose for upper
print(f"\nNumPy Cholesky (upper):\n{R_np.round(6)}")

### Cholesky Implementation

In [ ]:
def cholesky_upper(A):
    """Cholesky factorization A = R^T R, returns upper triangular R."""
    n = A.shape[0]
    R = np.zeros((n, n))
    for k in range(n):
        if A[k, k] <= 0:
            raise ValueError(f"Not SPD: a[{k},{k}] = {A[k,k]}")
        R[k, k] = np.sqrt(A[k, k])
        R[k, k+1:] = A[k, k+1:] / R[k, k]
        # Update Schur complement
        for j in range(k+1, n):
            A[j, k+1:] -= R[k, k+1:] * R[k, j]
    return R

# Test
np.random.seed(0)
B = np.random.randn(5, 5)
A = B.T @ B + np.eye(5)  # guaranteed SPD

R = cholesky_upper(A.copy())
print(f"||R^T R - A|| = {np.linalg.norm(R.T @ R - A):.2e}")
print(f"Diagonal of R: {R.diagonal().round(4)}  (all positive)")

### Cholesky as SPD Test

In [ ]:
# SPD matrix
A_spd = np.array([[4, 2], [2, 3]])
try:
    R = np.linalg.cholesky(A_spd)
    print(f"A = {A_spd.tolist()} -> SPD (Cholesky succeeded)")
except np.linalg.LinAlgError:
    print("Not SPD")

# Not SPD
A_bad = np.array([[1, 2], [2, 1]])
try:
    R = np.linalg.cholesky(A_bad)
    print(f"A = {A_bad.tolist()} -> SPD")
except np.linalg.LinAlgError:
    print(f"A = {A_bad.tolist()} -> NOT SPD (eigenvalues: {np.linalg.eigvalsh(A_bad).round(4)})")

# Symmetric but indefinite
A_indef = np.array([[1, 0, 0], [0, -2, 0], [0, 0, 3]])
try:
    R = np.linalg.cholesky(A_indef)
    print(f"Diagonal SPD")
except np.linalg.LinAlgError:
    print(f"Diagonal matrix with negative entry -> NOT SPD")

### Cost Comparison: Cholesky vs. LU vs. QR

In [ ]:
import time

sizes = [100, 200, 500, 1000, 2000]
t_chol = []
t_lu = []
t_qr = []

for n in sizes:
    B = np.random.randn(n, n)
    A = B.T @ B + n * np.eye(n)
    b = np.random.randn(n)

    t0 = time.perf_counter()
    R = np.linalg.cholesky(A)
    linalg.cho_solve((R, True), b)
    t_chol.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    np.linalg.solve(A, b)  # uses LU internally
    t_lu.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    Q, R_qr = np.linalg.qr(A)
    np.linalg.solve(R_qr, Q.T @ b)
    t_qr.append(time.perf_counter() - t0)

plt.figure(figsize=(8, 5))
plt.loglog(sizes, t_chol, 'go-', markersize=5, label='Cholesky ($n^3/3$)')
plt.loglog(sizes, t_lu, 'bo-', markersize=5, label='LU ($2n^3/3$)')
plt.loglog(sizes, t_qr, 'rs-', markersize=5, label='QR ($4n^3/3$)')

ref = [(n / sizes[-1])**3 * t_chol[-1] for n in sizes]
plt.loglog(sizes, ref, 'k--', alpha=0.3, label='$O(n^3)$ ref')

plt.xlabel('$n$')
plt.ylabel('Time (s)')
plt.title('Solving SPD systems: Cholesky vs LU vs QR')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

### Cholesky Stability: No Growth

In [ ]:
np.random.seed(5)
n = 50
kappas = np.logspace(1, 14, 20)

err_chol = []
err_lu = []

for kappa in kappas:
    # Build SPD with prescribed kappa
    Q, _ = np.linalg.qr(np.random.randn(n, n))
    lam = np.logspace(0, -np.log10(kappa), n)
    A = Q @ np.diag(lam) @ Q.T  # SPD with eigenvalues = lam
    A = (A + A.T) / 2  # ensure exact symmetry

    x_true = np.random.randn(n)
    b = A @ x_true

    # Cholesky
    try:
        L_ch = np.linalg.cholesky(A)
        y = linalg.solve_triangular(L_ch, b, lower=True)
        x_ch = linalg.solve_triangular(L_ch.T, y)
        err_chol.append(np.linalg.norm(x_ch - x_true) / np.linalg.norm(x_true))
    except:
        err_chol.append(np.nan)

    # LU
    x_lu = np.linalg.solve(A, b)
    err_lu.append(np.linalg.norm(x_lu - x_true) / np.linalg.norm(x_true))

eps = np.finfo(float).eps

plt.figure(figsize=(8, 5))
plt.loglog(kappas, err_chol, 'go-', markersize=4, label='Cholesky')
plt.loglog(kappas, err_lu, 'bs-', markersize=4, label='LU (pivoted)')
plt.loglog(kappas, kappas * eps, 'k--', alpha=0.4, label=r'$\kappa \cdot \epsilon$')
plt.xlabel(r'$\kappa(A)$')
plt.ylabel('Relative error')
plt.title('Cholesky vs LU on SPD matrices')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

### $\kappa(R) = \sqrt{\kappa(A)}$ for Cholesky

In [ ]:
np.random.seed(8)
kappas_test = np.logspace(2, 14, 15)
kappa_R = []

for kap in kappas_test:
    Q, _ = np.linalg.qr(np.random.randn(20, 20))
    lam = np.logspace(0, -np.log10(kap), 20)
    A = Q @ np.diag(lam) @ Q.T
    A = (A + A.T) / 2
    L = np.linalg.cholesky(A)
    kappa_R.append(np.linalg.cond(L))

plt.figure(figsize=(7, 5))
plt.loglog(kappas_test, kappa_R, 'bo-', markersize=5, label=r'$\kappa(R)$ measured')
plt.loglog(kappas_test, np.sqrt(kappas_test), 'r--', label=r'$\sqrt{\kappa(A)}$')
plt.xlabel(r'$\kappa(A)$')
plt.ylabel(r'$\kappa(R)$')
plt.title(r'Cholesky factor: $\kappa(R) = \sqrt{\kappa(A)}$')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

### Symmetric Elimination ($LDL^T$)

In [ ]:
# Demonstrate symmetric LDL^T factorization
A = np.array([[33,  7, 12, 17],
              [ 7, 23, 17, 22],
              [12, 17, 13, 27],
              [17, 22, 27,  3]], dtype=float)

# SciPy's LDL^T
L_ldl, D_ldl, perm = linalg.ldl(A)
print("LDL^T factorization of a symmetric (not SPD) matrix:")
print(f"\nD (diagonal/block-diagonal):")
print(D_ldl.round(4))
print(f"\nEigenvalues of A: {np.linalg.eigvalsh(A).round(4)}")
print(f"(Not all positive -> not SPD -> Cholesky would fail)")
print(f"\n||A - L D L^T|| = {np.linalg.norm(A - L_ldl @ D_ldl @ L_ldl.T):.2e}")